# 05 — Regime-Specific LSTM Training (3 variants × 3 seeds)

Trains **3 variants × 2 regimes × 3 seeds = 18 runs** by invoking `src/train_LSTM_regime.py` with `--regime both` (so each invocation covers calm + volatile sequentially). Seed 42 of each variant does full Optuna per regime; seeds 43 / 44 reuse seed 42's JSON via `--fixed-hparams`, skipping Optuna entirely.

**Variant H is intentionally not here** — H is a single-LSTM architectural variant, no regime-split ensemble.

| Variant | LSTM features | HMM source | Checkpoints |
|---|---|---|---|
| **O** | stationary (5) | `hmm_winner_O.joblib` + `regime_probabilities_O.parquet` | `lstm_{calm,volatile}_O_seed{N}.pt` |
| **A** | + sentiment (7) | `hmm_winner.joblib` + `regime_probabilities.parquet` | `lstm_{calm,volatile}_seed{N}.pt` |
| **B** | + VIX family (10) | `hmm_winner_B.joblib` + `regime_probabilities_B.parquet` | `lstm_{calm,volatile}_B_seed{N}.pt` |

Per Design 2: each variant uses its **own** HMM's Viterbi labels for window filtering. Variant B's regime LSTMs see the VIX-informed regime labels, not variant A's.

**Per-regime pipeline per seed** (handled inside `train_LSTM_regime.py`):

1. Load splits + variant-specific regime probs.
2. For each regime (calm, volatile):
   - Build `RegimeWindowDataset` filtering windows by majority Viterbi state.
   - Run Optuna (seed 42) or skip via `--fixed-hparams` (seeds 43/44).
   - Retrain with best params, early-stopping on regime-filtered val.
   - Evaluate on full test set (needed for ensemble blending in nb 06).
   - Save `lstm_{regime}{suffix}_seed{N}.pt` + `_scaler.joblib`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path().resolve().parent
SCRIPT = REPO_ROOT / "src" / "train_LSTM_regime.py"
assert SCRIPT.exists(), f"Missing: {SCRIPT}"
sys.path.insert(0, str(REPO_ROOT))

import config

print(f"Script: {SCRIPT}")
print(f"Python: {sys.executable}")


## Prerequisite check

Verifies all 9 HMM artifacts from nb 03 exist before starting any training.


In [ ]:
required_artifacts = {
    "Variant O HMM winner":      config.MODELS_DIR / "hmm_winner_O.joblib",
    "Variant O HMM meta":        config.MODELS_DIR / "hmm_meta_O.joblib",
    "Variant O regime probs":    config.DATA_PROCESSED / "regime_probabilities_O.parquet",
    "Variant A HMM winner":      config.MODELS_DIR / "hmm_winner.joblib",
    "Variant A HMM meta":        config.MODELS_DIR / "hmm_meta.joblib",
    "Variant A regime probs":    config.DATA_PROCESSED / "regime_probabilities.parquet",
    "Variant B HMM winner":      config.MODELS_DIR / "hmm_winner_B.joblib",
    "Variant B HMM meta":        config.MODELS_DIR / "hmm_meta_B.joblib",
    "Variant B regime probs":    config.DATA_PROCESSED / "regime_probabilities_B.parquet",
}
missing = {k: str(v) for k, v in required_artifacts.items() if not v.exists()}
if missing:
    raise RuntimeError(
        "Missing HMM artifacts:\n" +
        "\n".join(f"  {k}: {v}" for k, v in missing.items()) +
        "\n\nRun nb 03 (including cells 10b and 10c) first."
    )
print("All 9 HMM artifacts present (O / A / B × {winner, meta, regime_probs}).")


## Configure variants × seeds


In [ ]:
VARIANTS = [
    {
        "name":           "O",
        "features":       config.LSTM_VARIANT_O_FEATURES,
        "output_suffix_base": "_O",
        "regime_probs":   config.DATA_PROCESSED / "regime_probabilities_O.parquet",
        "hmm_meta":       config.MODELS_DIR / "hmm_meta_O.joblib",
    },
    {
        "name":           "A",
        "features":       config.LSTM_VARIANT_A_FEATURES,
        "output_suffix_base": "",
        "regime_probs":   config.DATA_PROCESSED / "regime_probabilities.parquet",
        "hmm_meta":       config.MODELS_DIR / "hmm_meta.joblib",
    },
    {
        "name":           "B",
        "features":       config.LSTM_VARIANT_B_FEATURES,
        "output_suffix_base": "_B",
        "regime_probs":   config.DATA_PROCESSED / "regime_probabilities_B.parquet",
        "hmm_meta":       config.MODELS_DIR / "hmm_meta_B.joblib",
    },
]
SEEDS = [42, 43, 44]

for v in VARIANTS:
    base = v["output_suffix_base"] or "<default>"
    for s in SEEDS:
        sfx = v["output_suffix_base"] + f"_seed{s}"
        print(f"  variant {v['name']}  suffix_base={base:<10s}  seed {s} → lstm_{{calm,volatile}}{sfx}.pt")
print(f"\nTotal runs (one per (variant, seed), each covering calm+volatile): {len(VARIANTS) * len(SEEDS)}")


## Training sweep

Per variant × seed: one invocation of `train_LSTM_regime.py --regime both` (which trains both regimes back-to-back). Seed 42 does full Optuna per regime; seeds 43/44 load the seed-42 JSON via `--fixed-hparams`.

Expected wall time per *variant* (all 3 seeds combined): ~30 min Optuna (2 regimes × ~15 min each) + 4 × ~5 min retrain (2 regimes × seeds 43/44) ≈ 50 min on Colab GPU. Total for 3 variants: ~150 min.


In [ ]:
variant_outputs = {}
MODELS_DIR = config.MODELS_DIR
MODELS_DIR.mkdir(parents=True, exist_ok=True)


def _run_one_seed(args_list, label):
    cmd = [sys.executable, "-u", str(SCRIPT), *args_list]
    print("=" * 80)
    print(f"[{label}]")
    print("Command:", " ".join(cmd))
    print("-" * 80)
    lines = []
    proc = subprocess.Popen(
        cmd, cwd=str(REPO_ROOT),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    rc = proc.wait()
    return rc, "".join(lines)


for v in VARIANTS:
    vname = v["name"]
    hparams_json_path = MODELS_DIR / f"hparams_lstm_regime{v['output_suffix_base'] or '_A'}.json"
    variant_outputs[vname] = {"seeds": {}}

    for seed_idx, seed in enumerate(SEEDS):
        suffix_for_seed = f"{v['output_suffix_base']}_seed{seed}"
        label = f"variant {vname}  seed {seed}"
        args_list = [
            "--features", *v["features"],
            "--regime", "both",
            "--output-suffix", suffix_for_seed,
            "--seed", str(seed),
            "--regime-probs-path", str(v["regime_probs"]),
            "--hmm-meta-path", str(v["hmm_meta"]),
        ]
        if seed_idx > 0:
            if not hparams_json_path.exists():
                print(f"  [{label}]  SKIPPED — seed 42 didn't produce {hparams_json_path}")
                variant_outputs[vname]["seeds"][seed] = {"status": "skipped"}
                continue
            args_list += ["--fixed-hparams", str(hparams_json_path)]

        try:
            rc, text = _run_one_seed(args_list, label)
        except Exception as exc:
            print(f"[{label}]  EXCEPTION: {exc}")
            variant_outputs[vname]["seeds"][seed] = {"status": "exception", "error": str(exc)}
            continue

        if rc != 0:
            print(f"[{label}]  FAILED (rc={rc})")
            variant_outputs[vname]["seeds"][seed] = {"status": "failed", "return_code": rc, "output": text}
            if seed_idx == 0:
                print(f"  Variant {vname}: seed 42 failed — skipping seeds {SEEDS[1:]}")
                break
            continue

        variant_outputs[vname]["seeds"][seed] = {"status": "ok", "output": text}

        # After seed 42 succeeds, persist its JSON for reuse
        if seed_idx == 0:
            marker = "=== Regime-Specific LSTM Results ==="
            idx = text.find(marker)
            if idx != -1:
                try:
                    blob = text[idx + len(marker):].strip()
                    results = json.loads(blob)
                    hparams_json_path.write_text(json.dumps(results, indent=2))
                    print(f"  [seed 42]  hparams JSON saved → {hparams_json_path.name}")
                except json.JSONDecodeError as exc:
                    print(f"  [seed 42]  JSON parse failed: {exc}")


## Aggregate per-(variant × regime × seed) test metrics


In [ ]:
import numpy as np

marker = "=== Regime-Specific LSTM Results ==="
per_seed_results = {}

for vname, vdata in variant_outputs.items():
    per_seed_results[vname] = {}
    for seed, info in vdata["seeds"].items():
        if info.get("status") != "ok": continue
        text = info["output"]
        idx = text.find(marker)
        if idx == -1: continue
        try:
            per_seed_results[vname][seed] = json.loads(text[idx + len(marker):].strip())
        except json.JSONDecodeError:
            continue

# Long form: one row per (variant, seed, regime)
rows = []
for vname, seeds_dict in per_seed_results.items():
    for seed, vresults in seeds_dict.items():
        for regime, r in vresults.items():
            rows.append({
                "variant":     vname, "seed": seed, "regime": regime,
                "test_MSE":    r["test_metrics"]["MSE"],
                "test_MAE":    r["test_metrics"]["MAE"],
                "seq_len":     r["best_params"]["seq_len"],
                "hidden_size": r["best_params"]["hidden_size"],
                "n_train_windows": r["n_train_windows"],
            })
per_seed_df = pd.DataFrame(rows)
print("Per-(variant, seed, regime) test metrics:")
display(per_seed_df)

# Aggregate: mean ± std across seeds per (variant, regime)
agg_rows = []
for vname, seeds_dict in per_seed_results.items():
    if not seeds_dict: continue
    for regime in ["calm", "volatile"]:
        mses = [vr[regime]["test_metrics"]["MSE"]
                for vr in seeds_dict.values() if regime in vr]
        maes = [vr[regime]["test_metrics"]["MAE"]
                for vr in seeds_dict.values() if regime in vr]
        if not mses: continue
        agg_rows.append({
            "variant":   vname, "regime": regime, "n_seeds": len(mses),
            "MSE_mean":  np.mean(mses),
            "MSE_std":   np.std(mses, ddof=1) if len(mses) > 1 else 0.0,
            "MAE_mean":  np.mean(maes),
            "MAE_std":   np.std(maes, ddof=1) if len(maes) > 1 else 0.0,
        })
agg_df = pd.DataFrame(agg_rows).set_index(["variant", "regime"])
print("\nMean ± std across seeds, per (variant, regime):")
agg_df


## Summary (fill in after execution)

For the paper:
- **F2 (volatile-LSTM degeneracy):** look for volatile LSTMs across seeds/variants with `prediction_std` ≪ target std (~1e-4 vs ~4e-3). If F2 appears consistently, it's a structural claim. If one seed's volatile LSTM blows up differently than another's, that's consistent with the "F2 seed-dependent failure mode" documented in `discussion.md`.
- **Training window asymmetry:** variant B's regime LSTMs train on ~180 volatile-majority windows (per the F4 data-scarcity framing), vs O / A's ~160 (both are 4-5 % of the full train set). Variance across seeds may differ across variants if the smaller volatile-subset leads to more unstable Optuna.
- **DM significance testing** across variants: in nb 06 (within-variant) and nb 07 (cross-variant).


## Verify saved artifacts


In [ ]:
import torch, joblib

for v in VARIANTS:
    base = v["output_suffix_base"]
    print(f"[Variant {v['name']}]  base suffix='{base}'")
    for seed in SEEDS:
        suffix = f"{base}_seed{seed}"
        for regime in ["calm", "volatile"]:
            pt_path     = MODELS_DIR / f"lstm_{regime}{suffix}.pt"
            scaler_path = MODELS_DIR / f"lstm_{regime}{suffix}_scaler.joblib"
            if not pt_path.exists() or not scaler_path.exists():
                print(f"  seed {seed}  [{regime}]  MISSING")
                continue
            ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)
            print(f"  seed {seed}  [{regime}]  n_features={ckpt['n_features']}  hidden={ckpt['hyperparameters']['hidden_size']}")
    print()

print("Verification complete.")


## Saved artifacts

| Pattern | Count | Description |
|---|---|---|
| `lstm_{calm,volatile}_O_seed{42,43,44}.pt` | 6 | variant O regime LSTMs (3 seeds × 2 regimes) |
| `lstm_{calm,volatile}_seed{42,43,44}.pt` | 6 | variant A regime LSTMs |
| `lstm_{calm,volatile}_B_seed{42,43,44}.pt` | 6 | variant B regime LSTMs |
| `*_scaler.joblib` | 18 | Matching scaler for each `.pt` |
| `hparams_lstm_regime{,_O,_B}.json` | 3 | Seed 42's per-regime `best_params` per variant |

**Total: 18 LSTM checkpoints + 18 scalers + 3 hparams JSONs.**

Feed into nb 06 (ensemble invocations for each of 3 variants × 3 seeds = 9 ensemble runs) which produces `data/processed/seeds/test_predictions{_O,,_B}_seed{N}.parquet`.
